In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#Langsmith Tracking And Tracing
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [5]:
from langchain.document_loaders import WebBaseLoader
docs = WebBaseLoader("https://python.langchain.com/docs/introduction/").load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
doc_splits = splitter.split_documents(docs)

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/Users/tusharshinde/code/my/AI/agentic_n_gen_ai/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from langchain.vectorstores import FAISS
vectorstore_db = FAISS.from_documents(doc_splits, embeddings)

retriver = vectorstore_db.as_retriever()

In [10]:
resp = vectorstore_db.similarity_search("Deployment: Turn your LangGraph applications into production-ready APIs and Assistants with LangGraph Platform.")
print(resp[0].page_content)

Development: Build your applications using LangChain's open-source components and third-party integrations.
Use LangGraph to build stateful agents with first-class streaming and human-in-the-loop support.
Productionization: Use LangSmith to inspect, monitor and evaluate your applications, so that you can continuously optimize and deploy with confidence.
Deployment: Turn your LangGraph applications into production-ready APIs and Assistants with LangGraph Platform.



LangChain implements a standard interface for large language models and related
technologies, such as embedding models and vector stores, and integrates with
hundreds of providers. See the integrations page for
more.


In [12]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

llm = ChatGroq(model="gemma2-9b-it")

prompt = ChatPromptTemplate.from_template("""
    Answer user input with only the provide context.
    <context>
    {context}
    </context>
""")

document_chain = create_stuff_documents_chain(llm, prompt)


In [16]:
from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriver, document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x3041dbed0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer user input with only the provide context.\n    <context>\n    {context}\n    </context>\n'), additional_kwargs={})])
            | ChatGro

In [22]:
resp = retrieval_chain.invoke({"input": "LangChain implements a standard interface for large language models"})
print(resp['answer'])

LangChain implements a standard interface for large language models and related technologies, such as embedding models and vector stores, and integrates with hundreds of providers. See the integrations page for more. 



In [18]:
resp

{'input': 'Deployment: Turn your LangGraph applications into production-ready APIs and Assistants with LangGraph Platform.',
 'context': [Document(id='6604c872-f0de-428d-a1e3-4a8d887b913c', metadata={'source': 'https://python.langchain.com/docs/introduction/', 'title': 'Introduction | 🦜️🔗 LangChain', 'description': 'LangChain is a framework for developing applications powered by large language models (LLMs).', 'language': 'en'}, page_content="Development: Build your applications using LangChain's open-source components and third-party integrations.\nUse LangGraph to build stateful agents with first-class streaming and human-in-the-loop support.\nProductionization: Use LangSmith to inspect, monitor and evaluate your applications, so that you can continuously optimize and deploy with confidence.\nDeployment: Turn your LangGraph applications into production-ready APIs and Assistants with LangGraph Platform.\n\n\n\nLangChain implements a standard interface for large language models and rel

In [32]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Translate the following from English to Marathi"),
    HumanMessage(content="Hello, how are you?")
]

resp = llm.invoke(messages)
print(resp)

content='नमस्कार, कसे आहात? (Namaskar, kase aahate?) \n' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 22, 'total_tokens': 46, 'completion_time': 0.043636364, 'prompt_time': 0.002112271, 'queue_time': 0.016732287000000002, 'total_time': 0.045748635}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None} id='run--3750b118-647c-485e-ae7e-f1844fb8f76d-0' usage_metadata={'input_tokens': 22, 'output_tokens': 24, 'total_tokens': 46}


In [29]:
from langchain_community.llms import Ollama

ollama_llm = Ollama(model="llama3.2:latest")

resp = ollama_llm.invoke(messages)

In [31]:
print(resp)

Hello! How can I assist you today?

Here's a direct translation in Marathi:

नमस्ते (Namaste) - Hello
आपा काय आहेत? (Apa kaa ahete?) - How are you? 

Note: In Marathi culture, 'Namaste' is a common greeting that combines respect with the phrase for "hello".
